[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/073.%20The%20Vehicle%20Routing%20Problem%20with%20Time%20Windows%20%28VRPTW%29/P73-Tier-5_executed.ipynb)

---

# Problem 73: The Vehicle Routing Problem with Time Windows (VRPTW)

## Tier 5: An Integrated Digital Twin Demonstration

### Purpose and scope

This notebook builds a small, synthetic digital-twin-style simulation for a delivery operation. It brings together customer data, vehicle telemetry, sensor updates, operational indicators, and scenario testing. The code illustrates how these data streams can be organized and monitored over simulated time.

The notebook does **not** formulate or solve a VRPTW, construct delivery routes, or optimize vehicle assignments. It also does not implement a production digital twin, a calibrated traffic model, or a validated forecasting system. In particular, the reported predictive-accuracy series is a deliberately specified synthetic indicator, not accuracy measured against forecast outcomes. Scenario results are outputs of the stated toy simulation and should not be interpreted as empirical operational improvements.

### Inputs and outputs

- **Inputs:** six illustrative customers, three illustrative vehicles, six illustrative sensors, and a travel-time matrix based on Euclidean distance.
- **Simulation:** sensor values and customer and vehicle states are updated over a one-hour simulated period, with a one-second time step.
- **Outputs:** time-indexed operational indicators, event records, descriptive analytics, scenario summaries, and visualizations.

### Background: the VRPTW context

In a vehicle routing problem with time windows, each customer has a service time window, often written as $[a_i, b_i]$, where $a_i$ is the earliest service time and $b_i$ is the latest acceptable service time. A route must respect vehicle capacity and travel and service times while serving customers within their windows. A digital twin can provide operational data to support such planning, but monitoring and simulation alone do not constitute a routing solution.

This notebook uses customer ready times and due dates as minute-of-day values. The simulated clock is measured in seconds after midnight, so the two representations are converted when customer urgency is updated. The travel-time matrix is constructed and retained as instance data, but this demonstration does not use it to generate or evaluate routes.

### Modeling assumptions

- Customer locations are points in a two-dimensional coordinate system; straight-line distance is used only to construct the illustrative travel-time matrix.
- Sensor observations, vehicle movement, and real-time demand are synthetic. They are not connected to physical devices or external data feeds.
- Vehicle movement is a random-walk illustration, not road-network navigation. The traffic multiplier modifies simulated movement speed and sensor congestion readings; it does not modify a route plan.
- The configured synchronization accuracy is a scenario parameter, not a statistic measured by this simulation. System uptime is not estimated.
- Scenario metrics describe the simulation outputs. No delay reduction, cost saving, customer-satisfaction gain, or forecast performance is inferred without a defined and validated measurement process.

### Digital-twin workflow

1. **Represent the system:** define customers, vehicles, sensors, and shared instance data.
2. **Integrate sensor observations:** update each sensor at its specified interval and retain its latest observation between updates.
3. **Advance simulated states:** update vehicle telemetry, customer demand, and customer urgency.
4. **Monitor indicators and events:** calculate operational indicators at each time step and record selected threshold events.
5. **Analyze and visualize:** summarize the simulated indicators and compare explicitly defined scenarios.

A digital twin is useful when a model and a physical operation exchange sufficiently current and reliable data. The small simulation here demonstrates the organization of such a workflow; it does not establish the data quality, predictive validity, or optimization capability required for a deployed system.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import random
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("default")
sns.set_palette("husl")

# Fixed seeds make the synthetic data stream reproducible when cells run in order.
np.random.seed(42)
random.seed(42)

## 1. Represent the digital-twin instance

A data class groups related values into one named object. This makes the model easier to inspect than a collection of unrelated variables. Customers hold location, demand, service-window, and priority data; vehicles hold capacity and telemetry; sensors hold their measurement schedule and latest observation; and the instance connects these objects to the travel-time matrix.

The travel-time matrix entry $t_{ij}$ represents the illustrative travel time from location $i$ to location $j$. The code computes straight-line distance $d_{ij}$ and sets $t_{ij}=2.5d_{ij}$. This is a simple geometric approximation, not a road-network travel-time estimate.

In [ ]:
@dataclass
class DigitalTwinCustomer:
    """Customer record with static and simulated real-time attributes."""

    id: int
    x: float
    y: float
    demand: float
    ready_time: float
    due_date: float
    service_time: float
    real_time_demand: float = 0.0
    dynamic_window: Optional[Tuple[float, float]] = None
    priority_level: int = 1
    location_accuracy: float = 1.0


@dataclass
class DigitalTwinVehicle:
    """Vehicle record with simulated telemetry and operating status."""

    id: int
    capacity: float
    current_location: Tuple[float, float] = (0.0, 0.0)
    current_load: float = 0.0
    battery_level: float = 100.0
    status: str = "available"
    speed: float = 30.0
    fuel_efficiency: float = 8.0
    telemetry_timestamp: float = 0.0
    route_progress: float = 0.0


@dataclass
class IoTSensor:
    """Sensor record containing its update interval and latest data."""

    id: str
    type: str
    location: Tuple[float, float]
    accuracy: float
    update_frequency: float
    last_update: float = 0.0
    data: Optional[Dict[str, Any]] = None


@dataclass
class DigitalTwinInstance:
    """Collection of modeled entities and shared simulation parameters."""

    customers: List[DigitalTwinCustomer]
    vehicles: List[DigitalTwinVehicle]
    sensors: List[IoTSensor]
    travel_time_matrix: np.ndarray
    current_time: float = 0.0
    synchronization_accuracy: float = 0.997
    uptime: float = 0.0

In [ ]:
def create_digital_twin_instance() -> DigitalTwinInstance:
    """Create the illustrative customer, vehicle, sensor, and travel data."""
    customers = [
        DigitalTwinCustomer(
            id=1, x=10, y=15, demand=40, ready_time=480, due_date=600,
            service_time=15, real_time_demand=42, priority_level=2,
            location_accuracy=0.95,
        ),
        DigitalTwinCustomer(
            id=2, x=20, y=10, demand=35, ready_time=540, due_date=660,
            service_time=10, real_time_demand=33, priority_level=1,
            location_accuracy=0.98,
        ),
        DigitalTwinCustomer(
            id=3, x=15, y=25, demand=50, ready_time=840, due_date=960,
            service_time=20, real_time_demand=55, priority_level=3,
            location_accuracy=0.92,
        ),
        DigitalTwinCustomer(
            id=4, x=25, y=20, demand=30, ready_time=600, due_date=720,
            service_time=12, real_time_demand=28, priority_level=1,
            location_accuracy=0.97,
        ),
        DigitalTwinCustomer(
            id=5, x=12, y=8, demand=45, ready_time=720, due_date=840,
            service_time=18, real_time_demand=48, priority_level=2,
            location_accuracy=0.94,
        ),
        DigitalTwinCustomer(
            id=6, x=18, y=22, demand=38, ready_time=780, due_date=900,
            service_time=14, real_time_demand=35, priority_level=1,
            location_accuracy=0.96,
        ),
    ]

    vehicles = [
        DigitalTwinVehicle(
            id=1, capacity=100, current_location=(5, 5),
            battery_level=85, status="available",
        ),
        DigitalTwinVehicle(
            id=2, capacity=100, current_location=(15, 10),
            battery_level=92, status="in_route",
        ),
        DigitalTwinVehicle(
            id=3, capacity=100, current_location=(0, 0),
            battery_level=78, status="maintenance",
        ),
    ]

    sensors = [
        IoTSensor(
            id="GPS_001", type="gps", location=(0, 0),
            accuracy=0.5, update_frequency=1.0,
        ),
        IoTSensor(
            id="Traffic_001", type="traffic", location=(15, 15),
            accuracy=0.8, update_frequency=30.0,
        ),
        IoTSensor(
            id="Weather_001", type="weather", location=(10, 10),
            accuracy=0.9, update_frequency=300.0,
        ),
        IoTSensor(
            id="Demand_001", type="demand", location=(20, 20),
            accuracy=0.7, update_frequency=60.0,
        ),
        IoTSensor(
            id="Vehicle_001", type="vehicle_status", location=(0, 0),
            accuracy=0.95, update_frequency=5.0,
        ),
        IoTSensor(
            id="Traffic_002", type="traffic", location=(25, 25),
            accuracy=0.85, update_frequency=30.0,
        ),
    ]

    locations = [(0, 0)] + [(customer.x, customer.y) for customer in customers]
    travel_time_matrix = np.zeros((len(locations), len(locations)))

    for i, origin in enumerate(locations):
        for j, destination in enumerate(locations):
            if i != j:
                distance = np.sqrt(
                    (origin[0] - destination[0]) ** 2
                    + (origin[1] - destination[1]) ** 2
                )
                travel_time_matrix[i, j] = distance * 2.5

    return DigitalTwinInstance(
        customers=customers,
        vehicles=vehicles,
        sensors=sensors,
        travel_time_matrix=travel_time_matrix,
    )


digital_twin = create_digital_twin_instance()
print("Digital-twin instance created.")
print(f"  Customers: {len(digital_twin.customers)}")
print(f"  Vehicles: {len(digital_twin.vehicles)}")
print(f"  Sensors: {len(digital_twin.sensors)}")
print(
    "  Configured synchronization accuracy: "
    f"{digital_twin.synchronization_accuracy:.1%} (assumption, not measured)"
)

## 2. Simulate sensor updates and operational states

A simulation state is the collection of modeled values at one point in time. At each time step, the simulator checks whether each sensor is due to report, updates vehicle and customer attributes, calculates indicators, and records selected events. Sensor data are held between scheduled updates, as they would be in a stream-processing system that retains the latest known observation.

The state transition can be written abstractly as

$$x_{t+\Delta t}=F(x_t, y_t, \varepsilon_t),$$

where $x_t$ is the modeled system state, $y_t$ is the latest available sensor information, $\varepsilon_t$ represents synthetic random variation, and $F$ is the update procedure implemented by the simulator methods. This equation describes the simulation workflow; it is not a learned physical model.

The clock starts at 08:00, represented as seconds after midnight, so customer windows expressed in minutes after midnight can be compared consistently. A one-hour run advances the clock in one-second steps. Vehicle movement is intentionally only illustrative: moving vehicles take a small random step, with speed reduced according to the latest average simulated traffic congestion.

In [ ]:
class DigitalTwinSimulator:
    """Advance the synthetic digital-twin state and record operational indicators."""

    def __init__(
        self,
        instance: DigitalTwinInstance,
        traffic_multiplier: float = 1.0,
    ) -> None:
        if traffic_multiplier < 0:
            raise ValueError("traffic_multiplier must be non-negative")

        self.instance = instance
        self.current_time = 8 * 60 * 60
        self.time_step = 1.0
        self.max_simulation_time = 3600.0
        self.traffic_multiplier = traffic_multiplier
        self.kpi_history = defaultdict(list)
        self.event_log = []

    def update_sensor_data(self) -> Dict[str, Dict[str, Any]]:
        """Refresh due sensors and return the latest observation from each sensor."""
        sensor_data = {}

        for sensor in self.instance.sensors:
            is_due = (
                sensor.data is None
                or self.current_time - sensor.last_update >= sensor.update_frequency
            )

            if is_due:
                if sensor.type == "gps":
                    sensor.data = {
                        "timestamp": self.current_time,
                        "location_accuracy": sensor.accuracy,
                        "satellites": int(np.random.randint(8, 12)),
                    }
                elif sensor.type == "traffic":
                    base_congestion = 0.3 * self.traffic_multiplier
                    time_factor = 1.0 + 0.5 * np.sin(self.current_time / 300)
                    congestion = (
                        base_congestion
                        * time_factor
                        * np.random.uniform(0.8, 1.2)
                    )
                    congestion = float(np.clip(congestion, 0.0, 1.0))
                    sensor.data = {
                        "timestamp": self.current_time,
                        "congestion_level": congestion,
                        "average_speed": 30.0 * (1.0 - congestion),
                        "incidents": int(np.random.choice([0, 1], p=[0.95, 0.05])),
                    }
                elif sensor.type == "weather":
                    sensor.data = {
                        "timestamp": self.current_time,
                        "temperature": np.random.uniform(15, 25),
                        "humidity": np.random.uniform(40, 70),
                        "wind_speed": np.random.uniform(0, 15),
                        "visibility": np.random.uniform(5, 10),
                    }
                elif sensor.type == "demand":
                    seasonal_factor = 1.0 + 0.2 * np.sin(self.current_time / 600)
                    demand_factor = seasonal_factor * np.random.uniform(0.9, 1.1)
                    sensor.data = {
                        "timestamp": self.current_time,
                        "demand_multiplier": float(demand_factor),
                        "urgency_level": int(
                            np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])
                        ),
                    }
                elif sensor.type == "vehicle_status":
                    vehicle_count = len(self.instance.vehicles)
                    sensor.data = {
                        "timestamp": self.current_time,
                        "active_vehicles": sum(
                            vehicle.status == "in_route"
                            for vehicle in self.instance.vehicles
                        ),
                        "available_vehicles": sum(
                            vehicle.status == "available"
                            for vehicle in self.instance.vehicles
                        ),
                        "maintenance_vehicles": sum(
                            vehicle.status == "maintenance"
                            for vehicle in self.instance.vehicles
                        ),
                        "average_battery": float(
                            np.mean(
                                [
                                    vehicle.battery_level
                                    for vehicle in self.instance.vehicles
                                ]
                            )
                        ) if vehicle_count else 0.0,
                    }

                sensor.last_update = self.current_time

            if sensor.data is not None:
                sensor_data[sensor.id] = sensor.data

        return sensor_data

    def update_vehicle_states(self, sensor_data: Dict[str, Dict[str, Any]]) -> None:
        """Advance telemetry and illustrative movement for vehicles in transit."""
        traffic_values = [
            data["congestion_level"]
            for sensor_id, data in sensor_data.items()
            if sensor_id.startswith("Traffic_")
        ]
        congestion = float(np.mean(traffic_values)) if traffic_values else 0.0

        for vehicle in self.instance.vehicles:
            if vehicle.status == "in_route":
                vehicle.battery_level = max(0.0, vehicle.battery_level - 0.01)
                vehicle.route_progress += np.random.uniform(0.01, 0.03)

                if vehicle.current_location != (0.0, 0.0):
                    effective_speed = vehicle.speed * (1.0 - congestion)
                    movement = effective_speed / 3600.0 * self.time_step
                    angle = np.random.uniform(0, 2 * np.pi)
                    new_x = vehicle.current_location[0] + movement * np.cos(angle)
                    new_y = vehicle.current_location[1] + movement * np.sin(angle)
                    vehicle.current_location = (new_x, new_y)

            vehicle.telemetry_timestamp = self.current_time

    def update_customer_states(
        self,
        sensor_data: Dict[str, Dict[str, Any]],
    ) -> None:
        """Update synthetic demand and priority using the simulated clock."""
        demand_multiplier = 1.0
        demand_data = sensor_data.get("Demand_001")
        if demand_data is not None:
            demand_multiplier = demand_data.get("demand_multiplier", 1.0)

        time_of_day_minutes = self.current_time / 60.0

        for customer in self.instance.customers:
            customer.real_time_demand = (
                customer.demand
                * demand_multiplier
                * np.random.uniform(0.9, 1.1)
            )

            minutes_until_due = customer.due_date - time_of_day_minutes
            if minutes_until_due <= 60:
                customer.priority_level = 3
            elif minutes_until_due <= 120:
                customer.priority_level = 2
            else:
                customer.priority_level = 1

    def calculate_kpis(
        self,
        sensor_data: Dict[str, Dict[str, Any]],
    ) -> Dict[str, float]:
        """Calculate descriptive indicators from the current simulated state."""
        total_vehicles = len(self.instance.vehicles)
        active_vehicles = sum(
            vehicle.status == "in_route" for vehicle in self.instance.vehicles
        )
        vehicle_utilization = (
            active_vehicles / total_vehicles if total_vehicles else 0.0
        )

        battery_levels = [
            vehicle.battery_level for vehicle in self.instance.vehicles
        ]
        average_battery = float(np.mean(battery_levels)) if battery_levels else 0.0

        total_customers = len(self.instance.customers)
        high_priority_customers = sum(
            customer.priority_level == 3 for customer in self.instance.customers
        )
        service_pressure = (
            high_priority_customers / total_customers if total_customers else 0.0
        )

        demands = [
            customer.real_time_demand for customer in self.instance.customers
        ]
        average_demand = float(np.mean(demands)) if demands else 0.0
        demand_variance = float(np.var(demands)) if demands else 0.0

        congestion_values = [
            data["congestion_level"]
            for sensor_id, data in sensor_data.items()
            if sensor_id.startswith("Traffic_")
        ]
        average_congestion = (
            float(np.mean(congestion_values)) if congestion_values else 0.0
        )

        # This formula creates a synthetic indicator; no forecast is evaluated.
        synthetic_prediction_indicator = float(
            0.85 + 0.1 * np.sin(self.current_time / 600.0)
        )
        system_efficiency = 1.0 - demand_variance / (average_demand**2 + 1.0)

        return {
            "timestamp": self.current_time,
            "vehicle_utilization": float(vehicle_utilization),
            "average_battery": average_battery,
            "service_pressure": float(service_pressure),
            "average_demand": average_demand,
            "demand_variance": demand_variance,
            "average_congestion": average_congestion,
            "synthetic_prediction_indicator": synthetic_prediction_indicator,
            "system_efficiency": float(system_efficiency),
        }

    def run_simulation(self) -> Dict[str, Any]:
        """Run the simulation and return its recorded indicators and events."""
        print("Starting the digital-twin simulation.")
        print(f"Simulation duration: {self.max_simulation_time:.0f} seconds")
        print(f"Time step: {self.time_step:.0f} second")

        start_time = time.time()
        end_time = self.current_time + self.max_simulation_time
        final_kpis = {}

        while self.current_time < end_time:
            sensor_data = self.update_sensor_data()
            self.update_vehicle_states(sensor_data)
            self.update_customer_states(sensor_data)
            final_kpis = self.calculate_kpis(sensor_data)

            for key, value in final_kpis.items():
                self.kpi_history[key].append(value)

            if final_kpis["service_pressure"] > 0.5:
                self.event_log.append(
                    {
                        "timestamp": self.current_time,
                        "type": "high_service_pressure",
                        "value": final_kpis["service_pressure"],
                    }
                )

            if final_kpis["average_battery"] < 20.0:
                self.event_log.append(
                    {
                        "timestamp": self.current_time,
                        "type": "low_battery_warning",
                        "value": final_kpis["average_battery"],
                    }
                )

            self.current_time += self.time_step

        execution_time = time.time() - start_time
        print(f"Simulation completed in {execution_time:.2f} seconds.")
        print(f"Events recorded: {len(self.event_log)}")

        return {
            "execution_time": execution_time,
            "total_events": len(self.event_log),
            "kpi_history": dict(self.kpi_history),
            "event_log": self.event_log,
            "final_kpis": final_kpis,
        }


simulator = DigitalTwinSimulator(digital_twin)
simulation_results = simulator.run_simulation()

## 3. Describe the simulated indicators

A time series is a sequence of values recorded at successive times. This section summarizes the synthetic indicator and operational series; it does not assess a predictive model. Forecast accuracy requires both a forecast and a corresponding observed outcome, neither of which is defined for the synthetic prediction indicator here.

The demand coefficient of variation, $s_D/\bar{D}$, expresses demand standard deviation $s_D$ relative to mean demand $\bar{D}$. It is a measure of relative variability, not a forecast error. The correlation coefficient between simulated congestion and service pressure is a descriptive association only. It does not show that congestion causes service pressure or that either variable has been predicted accurately. If either series is constant, the correlation is undefined and is reported as `NaN`.

In [ ]:
def analyze_predictive_analytics(kpi_history: Dict[str, List[float]]) -> Dict[str, float]:
    """Summarize synthetic indicators and descriptive operational associations."""
    required_keys = {
        "average_demand",
        "average_congestion",
        "service_pressure",
        "synthetic_prediction_indicator",
    }
    missing_keys = required_keys.difference(kpi_history)
    if missing_keys:
        raise KeyError(f"Missing KPI history fields: {sorted(missing_keys)}")

    indicator = np.asarray(
        kpi_history["synthetic_prediction_indicator"], dtype=float
    )
    demand = np.asarray(kpi_history["average_demand"], dtype=float)
    congestion = np.asarray(kpi_history["average_congestion"], dtype=float)
    service_pressure = np.asarray(kpi_history["service_pressure"], dtype=float)

    if not indicator.size or not demand.size:
        raise ValueError("KPI histories must contain at least one observation")

    mean_demand = float(np.mean(demand))
    demand_coefficient_of_variation = (
        float(np.std(demand) / abs(mean_demand))
        if not np.isclose(mean_demand, 0.0)
        else float("nan")
    )

    if (
        congestion.size == service_pressure.size
        and congestion.size > 1
        and not np.isclose(np.std(congestion), 0.0)
        and not np.isclose(np.std(service_pressure), 0.0)
    ):
        congestion_service_correlation = float(
            np.corrcoef(congestion, service_pressure)[0, 1]
        )
    else:
        congestion_service_correlation = float("nan")

    analysis = {
        "indicator_mean": float(np.mean(indicator)),
        "indicator_std": float(np.std(indicator)),
        "indicator_min": float(np.min(indicator)),
        "indicator_max": float(np.max(indicator)),
        "demand_coefficient_of_variation": demand_coefficient_of_variation,
        "congestion_service_correlation": congestion_service_correlation,
    }

    print("Descriptive analysis of simulated indicators")
    print(
        "  Synthetic indicator mean: "
        f"{analysis['indicator_mean']:.3f}"
    )
    print(
        "  Synthetic indicator standard deviation: "
        f"{analysis['indicator_std']:.3f}"
    )
    print(
        "  Demand coefficient of variation: "
        f"{analysis['demand_coefficient_of_variation']:.3f}"
    )
    print(
        "  Congestion-service-pressure correlation: "
        f"{analysis['congestion_service_correlation']:.3f}"
    )
    print("  These summaries do not measure forecast accuracy.")

    return analysis


predictive_analysis = analyze_predictive_analytics(
    simulation_results["kpi_history"]
)

## 4. Compare explicitly defined operating scenarios

Scenario analysis changes selected inputs while retaining the same simulation structure. Here, the holiday-peak case uses the stated 45% demand increase, 20% traffic multiplier increase, and two maintenance assignments. A severe-disruption case is also included as a separate illustrative stress case.

For each case, demand is scaled, selected vehicles are marked as undergoing maintenance, and the simulator runs for 30 minutes. The reported values are final simulated utilization, service pressure, efficiency, and congestion. They are not estimates of delay, cost savings, or service-window compliance because this simulation does not model route completion or measure those outcomes.

The same random seed initializes the notebook's random streams, but the scenarios consume random numbers sequentially. Therefore, the scenario table is a descriptive demonstration, not a controlled causal experiment.

In [ ]:
def scenario_analysis() -> Dict[str, List[Dict[str, Any]]]:
    """Run the baseline, holiday-peak, and severe-disruption simulations."""
    scenarios = [
        {
            "name": "Baseline",
            "demand_increase": 0.0,
            "traffic_increase": 0.0,
            "maintenance_events": 0,
            "description": "Reference operating assumptions",
        },
        {
            "name": "Holiday Peak",
            "demand_increase": 0.45,
            "traffic_increase": 0.20,
            "maintenance_events": 2,
            "description": (
                "45% demand increase, 20% traffic multiplier increase, "
                "and two maintenance assignments"
            ),
        },
        {
            "name": "Severe Disruption",
            "demand_increase": 0.30,
            "traffic_increase": 0.50,
            "maintenance_events": 3,
            "description": (
                "30% demand increase, 50% traffic multiplier increase, "
                "and three maintenance assignments"
            ),
        },
    ]

    results = []

    for scenario in scenarios:
        instance = create_digital_twin_instance()

        for customer in instance.customers:
            customer.demand *= 1.0 + scenario["demand_increase"]
            customer.real_time_demand *= 1.0 + scenario["demand_increase"]

        maintenance_count = min(
            scenario["maintenance_events"], len(instance.vehicles)
        )
        for vehicle in instance.vehicles[:maintenance_count]:
            vehicle.status = "maintenance"

        scenario_simulator = DigitalTwinSimulator(
            instance,
            traffic_multiplier=1.0 + scenario["traffic_increase"],
        )
        scenario_simulator.max_simulation_time = 1800.0
        scenario_result = scenario_simulator.run_simulation()
        final_kpis = scenario_result["final_kpis"]

        result = {
            "name": scenario["name"],
            "description": scenario["description"],
            "vehicle_utilization": final_kpis["vehicle_utilization"],
            "service_pressure": final_kpis["service_pressure"],
            "system_efficiency": final_kpis["system_efficiency"],
            "average_congestion": final_kpis["average_congestion"],
            "average_demand": final_kpis["average_demand"],
            "total_events": scenario_result["total_events"],
        }
        results.append(result)

        print(f"\nScenario: {result['name']}")
        print(f"  Description: {result['description']}")
        print(
            "  Final vehicle utilization: "
            f"{result['vehicle_utilization']:.3f}"
        )
        print(f"  Final service pressure: {result['service_pressure']:.3f}")
        print(f"  Final system efficiency: {result['system_efficiency']:.3f}")
        print(f"  Final average congestion: {result['average_congestion']:.3f}")
        print(f"  Events recorded: {result['total_events']}")

    return {"scenarios": results}


scenario_results = scenario_analysis()

## 5. Visualize the simulation outputs

The plots show how the recorded indicators vary over the simulated hour and how final scenario indicators compare. The time axis is displayed in minutes after midnight, matching the customer-window convention. Scenario metrics use separate axes where their scales differ; the labels and legends identify each quantity and its units or dimensionless scale.

A plotted association or difference is descriptive. It does not establish causation, successful route optimization, or real-world performance.

In [ ]:
def visualize_digital_twin_results(
    sim_results: Dict[str, Any],
    pred_analysis: Dict[str, float],
    scen_results: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Plot simulated time series and scenario indicators."""
    kpi_history = sim_results["kpi_history"]
    timestamps_minutes = np.asarray(kpi_history["timestamp"]) / 60.0

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    ax1, ax2, ax3, ax4 = axes.flat

    ax1.plot(
        timestamps_minutes,
        kpi_history["vehicle_utilization"],
        label="Vehicle utilization",
        linewidth=1.8,
    )
    ax1.plot(
        timestamps_minutes,
        kpi_history["service_pressure"],
        label="Service pressure",
        linewidth=1.8,
    )
    ax1.plot(
        timestamps_minutes,
        kpi_history["system_efficiency"],
        label="System efficiency",
        linewidth=1.8,
    )
    ax1.set_xlabel("Time (minutes after midnight)")
    ax1.set_ylabel("Indicator value (dimensionless)")
    ax1.set_title("Simulated operational indicators")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.plot(
        timestamps_minutes,
        kpi_history["synthetic_prediction_indicator"],
        color="purple",
        linewidth=1.8,
        label="Synthetic indicator",
    )
    ax2.axhline(
        y=pred_analysis["indicator_mean"],
        color="black",
        linestyle="--",
        label=f"Mean: {pred_analysis['indicator_mean']:.3f}",
    )
    ax2.set_xlabel("Time (minutes after midnight)")
    ax2.set_ylabel("Indicator value (dimensionless)")
    ax2.set_title("Synthetic indicator (not measured forecast accuracy)")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    scenarios = scen_results["scenarios"]
    scenario_names = [scenario["name"] for scenario in scenarios]
    x = np.arange(len(scenario_names))
    width = 0.36

    ax3_twin = ax3.twinx()
    utilization_bars = ax3.bar(
        x - width / 2,
        [scenario["vehicle_utilization"] for scenario in scenarios],
        width,
        label="Vehicle utilization",
        color="tab:blue",
        alpha=0.75,
    )
    pressure_bars = ax3.bar(
        x + width / 2,
        [scenario["service_pressure"] for scenario in scenarios],
        width,
        label="Service pressure",
        color="tab:green",
        alpha=0.75,
    )
    congestion_line = ax3_twin.plot(
        x,
        [scenario["average_congestion"] for scenario in scenarios],
        color="tab:red",
        marker="o",
        linewidth=1.8,
        label="Average congestion",
    )
    ax3.set_xlabel("Scenario")
    ax3.set_ylabel("Utilization and pressure (dimensionless)")
    ax3_twin.set_ylabel("Average congestion (dimensionless)")
    ax3.set_title("Final simulated scenario indicators")
    ax3.set_xticks(x)
    ax3.set_xticklabels(scenario_names, rotation=10)
    ax3.set_ylim(0, 1)
    ax3_twin.set_ylim(0, 1)
    ax3.grid(True, axis="y", alpha=0.3)
    ax3.legend(
        [utilization_bars, pressure_bars, congestion_line[0]],
        ["Vehicle utilization", "Service pressure", "Average congestion"],
        loc="upper left",
        fontsize=8,
    )

    ax4.axis("off")
    architecture_text = (
        "Digital-twin demonstration\n"
        "\n"
        f"Customers: {len(digital_twin.customers)}\n"
        f"Vehicles: {len(digital_twin.vehicles)}\n"
        f"Sensors: {len(digital_twin.sensors)}\n"
        f"Traffic sensors: {sum(sensor.type == 'traffic' for sensor in digital_twin.sensors)}\n"
        f"Configured synchronization parameter: {digital_twin.synchronization_accuracy:.1%}\n"
        f"Simulation execution time: {sim_results['execution_time']:.2f} seconds\n"
        f"Recorded events: {sim_results['total_events']}\n"
        f"Synthetic indicator mean: {pred_analysis['indicator_mean']:.3f}\n"
        "\n"
        "Scope limits\n"
        "No route construction or VRPTW optimization\n"
        "No measured forecast accuracy or uptime\n"
        "No empirical delay or cost estimates"
    )
    ax4.text(
        0.04,
        0.95,
        architecture_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
    )

    fig.suptitle("Digital-twin simulation overview", fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_digital_twin_results(
    simulation_results,
    predictive_analysis,
    scenario_results,
)

## Summary and conclusions

This notebook demonstrates a small digital-twin-style simulation workflow: it represents customers, vehicles, and sensors; updates synthetic sensor observations on different schedules; advances simulated customer and vehicle states; records operational indicators and threshold events; and summarizes scenario runs.

### What the results can support

- The time series show the behavior of the indicators defined in the code under the stated synthetic assumptions.
- The scenario outputs show how those same indicators behave when the specified demand, traffic, and maintenance inputs are changed.
- The sensor update intervals illustrate how observations with different reporting frequencies can be combined while retaining the latest available value.

### What the results do not establish

- The notebook does not solve the vehicle routing problem with time windows. No customer sequence, feasible route, vehicle assignment, routing objective, or routing constraint is optimized.
- The synthetic prediction indicator is generated directly by a formula. Its mean and variation are summaries of that formula, not measured predictive performance.
- The configured synchronization accuracy is an input assumption, not an observed synchronization result; uptime is not calculated.
- The movement and congestion rules are illustrative and are not calibrated to road, vehicle, or sensor data.
- Delay, time-window compliance, customer satisfaction, travel-time reduction, and financial savings are not calculated by the model.

### Connection to a VRPTW digital twin

Operational data from a validated digital twin could inform a VRPTW model by updating travel times, customer demand, vehicle availability, or time-window information. A routing method would then need to use those inputs to construct and evaluate feasible routes. That optimization step lies outside this simulation, so the notebook should be read as a systems-monitoring demonstration rather than a routing solution.

### Further study

A natural next stage would be to connect the monitoring layer to validated operational data and clearly defined performance measures. Any routing optimization, calibration, or forecasting model would require its own explicit assumptions, mathematical formulation, and evaluation; none is implemented here.